In [7]:
# EGX360 — THE DEEP QUANT MODEL (Forward 5D Mean Target)
# Enhancing Trend Detection & Bias Control in the Egyptian Stock Exchange

import tensorflow as tf
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, KFold
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import joblib
import warnings
import os

warnings.filterwarnings('ignore')

print("🚀 Loading Final Dataset (EGX + USD + Gold)...")
current_dir = os.getcwd()

# ==========================================
# 1. Data Loading & Leakage Fix
# ==========================================
file_path = os.path.join(current_dir, "data", "EGX30_Final_v9.csv")
df = pd.read_csv(file_path)
df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None).dt.normalize()
df.set_index('timestamp', inplace=True)

ir_path = os.path.join(current_dir, "data", "cbe_interest_rate.csv")
if os.path.exists(ir_path):
    ir_df = pd.read_csv(ir_path)
    ir_df['Date'] = pd.to_datetime(ir_df['Date']).dt.normalize()
    ir_df.set_index('Date', inplace=True)
    df = df.join(ir_df, how='left')
    
    # ffill فقط لمنع تسريب المستقبل
    df['Interest_Rate'] = df['Interest_Rate'].ffill()
    df['IR_Change'] = df['Interest_Rate'].diff().fillna(0)

# حذف الأيام القديمة اللي ملهاش بيانات فائدة
df.dropna(subset=['Interest_Rate'], inplace=True)

# ==========================================
# 2. Feature Engineering (Quant Indicators)
# ==========================================
df['log_ret'] = np.log((df['close'] + 1e-6) / (df['close'].shift(1) + 1e-6))
df['price_velocity'] = df['log_ret'].diff()

if 'close_usd' in df.columns:
    df['log_ret_usd'] = np.log((df['close_usd'] + 1e-6) / (df['close_usd'].shift(1) + 1e-6))
    df['price_velocity_usd'] = df['log_ret_usd'].diff()

df['Volume_SMA_50'] = df['volume'].rolling(window=50).mean()
df['RVOL_50'] = (df['volume'] / (df['Volume_SMA_50'] + 1e-9)).clip(upper=5.0)

df['day_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 7)
df['day_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 7)

for period in [9, 21, 50]:
    ema_col = f'EMA_{period}'
    df[ema_col] = df['close'].ewm(span=period).mean()
    df[f'dist_EMA_{period}'] = (df['close'] - df[ema_col]) / (df[ema_col] + 1e-9)

delta = df['close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
df['RSI'] = 100 - (100 / (1 + gain/(loss + 1e-9)))

macd = df['close'].ewm(span=12).mean() - df['close'].ewm(span=26).mean()
df['MACD_Hist'] = macd - macd.ewm(span=9).mean()

tr = pd.concat([df['high'] - df['low'], 
                np.abs(df['high'] - df['close'].shift()), 
                np.abs(df['low'] - df['close'].shift())], axis=1).max(axis=1)
df['ATR_pct'] = tr.rolling(14).mean() / (df['close'] + 1e-9)

# ==========================================
# 3. Target Engineering (The Forward 5D Mean)
# ==========================================
print("🎯 Engineering Forward Target (Zero-Lag)...")
# حساب متوسط السعر للـ 5 أيام القادمة
df['Future_5D_Mean'] = df['close'].rolling(window=5).mean().shift(-5)

# الهدف: هل متوسط المستقبل أعلى من السعر الحالي؟
df['Target'] = (df['Future_5D_Mean'] > df['close']).astype(int)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=['Future_5D_Mean', 'Target'], inplace=True)
df.dropna(inplace=True)

final_features = [
    'log_ret', 'price_velocity', 'RVOL_50', 
    'day_sin', 'day_cos', 
    'dist_EMA_9', 'dist_EMA_21', 'dist_EMA_50', 
    'RSI', 'MACD_Hist', 
    'Interest_Rate', 'IR_Change', 'ATR_pct'
]
X = df[final_features].values
y = df['Target'].values

# ==========================================
# 4. ML Validation & Stacking (Strict Split)
# ==========================================
split = int(len(X) * 0.80)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

import optuna
tscv = TimeSeriesSplit(n_splits=5)

def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state': 42,
        'n_jobs': -1
    }
    model = xgb.XGBClassifier(**params)
    score = cross_val_score(model, X_train_scaled, y_train, cv=tscv, scoring='accuracy').mean()
    return score

print("🚀 Tuning XGBoost...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=20) 
best_xgb_params = study_xgb.best_params

def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    model = lgb.LGBMClassifier(**params)
    score = cross_val_score(model, X_train_scaled, y_train, cv=tscv, scoring='accuracy').mean()
    return score

print("🚀 Tuning LightGBM...")
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=20)
best_lgb_params = study_lgb.best_params

print("\n⚙️ Training Final Stacked Model...")
best_xgb = xgb.XGBClassifier(**best_xgb_params, random_state=42, n_jobs=-1)
best_lgb = lgb.LGBMClassifier(**best_lgb_params, random_state=42, n_jobs=-1, verbose=-1)

base_models = [('xgb', best_xgb), ('lgbm', best_lgb)]
final_logic = LogisticRegression(C=1.0)

# KFold بدون خلط لحل مشكلة Stacking
kf_no_shuffle = KFold(n_splits=5, shuffle=False)
stack_model = StackingClassifier(estimators=base_models, final_estimator=final_logic, cv=kf_no_shuffle)
stack_model.fit(X_train_scaled, y_train)

# ==========================================
# 5. Final Evaluation (The Reality Check)
# ==========================================
y_pred = stack_model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred) * 100

print(f"\nREAL QUANT ACCURACY (Forward 5D Target): {acc:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Down Trend (0)', 'Up Trend (1)']))

# ==========================================
# 6. Strategy Backtesting
# ==========================================
test_dates = pd.Series(df.index[split:]).reset_index(drop=True)
test_close = df['close'].iloc[split:].reset_index(drop=True)

results_df = pd.DataFrame({
    'timestamp': test_dates,
    'close': test_close,
    'Actual_Trend': y_test,
    'Predicted_Trend': y_pred
})

initial_capital = 10000.0
results_df['daily_return'] = results_df['close'].pct_change()

# Shift signal by 1 to prevent look-ahead bias
results_df['signal'] = results_df['Predicted_Trend'].shift(1)
results_df['strategy_return'] = results_df['daily_return'] * results_df['signal']

bt_df = results_df.dropna().copy()
bt_df['cumulative_market'] = (1 + bt_df['daily_return']).cumprod()
bt_df['cumulative_strategy'] = (1 + bt_df['strategy_return']).cumprod()

bt_df['portfolio_value'] = initial_capital * bt_df['cumulative_strategy']
bt_df['bh_value'] = initial_capital * bt_df['cumulative_market']

final_portfolio = bt_df['portfolio_value'].iloc[-1]
final_bh = bt_df['bh_value'].iloc[-1]
strategy_profit_pct = ((final_portfolio - initial_capital) / initial_capital) * 100
market_profit_pct = ((final_bh - initial_capital) / initial_capital) * 100

print("-" * 50)
print(f"1) Buy & Hold (Market Baseline): {final_bh:,.2f} EGP ({market_profit_pct:.2f}%)")
print(f"2) EGX360 Strategy (Our Model):  {final_portfolio:,.2f} EGP ({strategy_profit_pct:.2f}%)")
print("-" * 50)

[I 2026-03-26 02:27:50,316] A new study created in memory with name: no-name-26933eaa-0414-4338-a239-bd3b73697823


🚀 Loading Final Dataset (EGX + USD + Gold)...
🎯 Engineering Forward Target (Zero-Lag)...
🚀 Tuning XGBoost...


[I 2026-03-26 02:27:52,060] Trial 0 finished with value: 0.5483516483516484 and parameters: {'n_estimators': 432, 'max_depth': 5, 'learning_rate': 0.02596077792979904, 'subsample': 0.7843373866887852, 'colsample_bytree': 0.8182990178435339, 'gamma': 2.580828725028863}. Best is trial 0 with value: 0.5483516483516484.
[I 2026-03-26 02:27:53,144] Trial 1 finished with value: 0.5410989010989011 and parameters: {'n_estimators': 186, 'max_depth': 6, 'learning_rate': 0.027214242234776267, 'subsample': 0.9896266910891801, 'colsample_bytree': 0.747871824255978, 'gamma': 0.8601012376765138}. Best is trial 0 with value: 0.5483516483516484.
[I 2026-03-26 02:27:55,042] Trial 2 finished with value: 0.5441758241758242 and parameters: {'n_estimators': 166, 'max_depth': 7, 'learning_rate': 0.016269149788955103, 'subsample': 0.94741531530384, 'colsample_bytree': 0.7838762772849024, 'gamma': 2.45339751291632}. Best is trial 0 with value: 0.5483516483516484.
[I 2026-03-26 02:27:57,145] Trial 3 finished wi

🚀 Tuning LightGBM...


[I 2026-03-26 02:28:13,562] Trial 0 finished with value: 0.5446153846153846 and parameters: {'n_estimators': 102, 'max_depth': 6, 'learning_rate': 0.026694279486966732, 'subsample': 0.8488491934722961, 'colsample_bytree': 0.7808331442121332, 'num_leaves': 42}. Best is trial 0 with value: 0.5446153846153846.
[I 2026-03-26 02:28:14,339] Trial 1 finished with value: 0.5487912087912088 and parameters: {'n_estimators': 406, 'max_depth': 4, 'learning_rate': 0.014781488694470038, 'subsample': 0.6135746792672473, 'colsample_bytree': 0.6166957526126553, 'num_leaves': 49}. Best is trial 1 with value: 0.5487912087912088.
[I 2026-03-26 02:28:15,472] Trial 2 finished with value: 0.5316483516483516 and parameters: {'n_estimators': 233, 'max_depth': 7, 'learning_rate': 0.07406267753826835, 'subsample': 0.7444102508151529, 'colsample_bytree': 0.893182183346048, 'num_leaves': 54}. Best is trial 1 with value: 0.5487912087912088.
[I 2026-03-26 02:28:16,102] Trial 3 finished with value: 0.5426373626373626


⚙️ Training Final Stacked Model...

REAL QUANT ACCURACY (Forward 5D Target): 59.00%

Classification Report:
                precision    recall  f1-score   support

Down Trend (0)       0.51      0.35      0.41       569
  Up Trend (1)       0.62      0.76      0.68       797

      accuracy                           0.59      1366
     macro avg       0.57      0.56      0.55      1366
  weighted avg       0.58      0.59      0.57      1366

--------------------------------------------------
1) Buy & Hold (Market Baseline): 44,907.77 EGP (349.08%)
2) EGX360 Strategy (Our Model):  61,661.86 EGP (516.62%)
--------------------------------------------------


In [ ]:
# ==============================================================================
# EGX360 — THE DEEP QUANT MODEL (ULTIMATE REALITY VERSION)
# Blending Trend Mechanics (EMA) with Forward Returns & Bias Control
# ==============================================================================

import tensorflow as tf
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, KFold
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller
import joblib
import warnings
import os

warnings.filterwarnings('ignore')

# 1. Environment Setup & Initialization
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"Success! GPU found: {gpu_devices}")
else:
    print("Running on CPU.")

print("🚀 Loading Final Dataset (EGX + USD + Gold)...")
current_dir = os.getcwd()

# ==========================================
# 2. Data Loading & Leakage Fix
# ==========================================
file_path = os.path.join(current_dir, "data", "EGX30_Final_v9.csv")
df = pd.read_csv(file_path)
df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None).dt.normalize()
df.set_index('timestamp', inplace=True)

ir_path = os.path.join(current_dir, "data", "cbe_interest_rate.csv")
if os.path.exists(ir_path):
    ir_df = pd.read_csv(ir_path)
    ir_df['Date'] = pd.to_datetime(ir_df['Date']).dt.normalize()
    ir_df.set_index('Date', inplace=True)
    df = df.join(ir_df, how='left')
    
    # ffill فقط لمنع تسريب المستقبل (Zero Leakage)
    df['Interest_Rate'] = df['Interest_Rate'].ffill()
    df['IR_Change'] = df['Interest_Rate'].diff().fillna(0)
    print("Interest Rates Merged Correctly.")

df.dropna(subset=['Interest_Rate'], inplace=True)
df.sort_index(inplace=True)

# حساب العوائد اللوغاريتمية مبكراً للرسومات
df['log_ret'] = np.log((df['close'] + 1e-6) / (df['close'].shift(1) + 1e-6))

# ==========================================
# 3. Raw Market Visualization & Proofs (User's Original EDA)
# ==========================================
print("\n📊 Step 1: Visualizing EGX30 vs Macroeconomic Drivers...")
fig_macro = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                          subplot_titles=('EGX30 Close Price', 'USD/EGP Exchange Rate', 'Gold Price (EGP)'))
fig_macro.add_trace(go.Scatter(x=df.index, y=df['close'], mode='lines', name='EGX30', line=dict(color='black')), row=1, col=1)
if 'usd_egp_rate' in df.columns:
    fig_macro.add_trace(go.Scatter(x=df.index, y=df['usd_egp_rate'], mode='lines', name='USD/EGP', line=dict(color='green')), row=2, col=1)
if 'gold_egp' in df.columns:
    fig_macro.add_trace(go.Scatter(x=df.index, y=df['gold_egp'], mode='lines', name='Gold (EGP)', line=dict(color='goldenrod')), row=3, col=1)
fig_macro.update_layout(height=800, title_text="Macroeconomic Trends in Egypt", template='plotly_white', hovermode='x unified')
fig_macro.show()

print("\n📊 Step 2: Statistical Properties of Returns (Market Noise)...")
returns = df['log_ret'].dropna()
fig, axes = plt.subplots(3, 1, figsize=(12, 14))
fig.tight_layout(pad=6.0)

# Fat Tails
sns.histplot(returns, bins=100, kde=True, stat="density", ax=axes[0], color='blue', alpha=0.5, label='Actual EGX Returns')
mu, std = stats.norm.fit(returns)
xmin, xmax = axes[0].get_xlim()
x = np.linspace(xmin, xmax, 100)
axes[0].plot(x, stats.norm.pdf(x, mu, std), 'k', linewidth=2, linestyle='dashed', label='Normal Dist')
axes[0].set_title("Return Distribution: The 'Fat Tails' Problem", fontsize=12, fontweight='bold')
axes[0].legend()

# Volatility Clustering
axes[1].plot(returns.index, np.abs(returns), color='red', alpha=0.7, linewidth=0.8)
axes[1].set_title("Volatility Clustering: Crises and Panics Group Together", fontsize=12, fontweight='bold')

# Autocorrelation
plot_acf(returns, lags=40, ax=axes[2], alpha=0.05, color='green')
axes[2].set_title("Autocorrelation (ACF): Proof of Market Noise", fontsize=12, fontweight='bold')
plt.show()

print("\n📊 Step 3: Extreme Events & Outliers Analysis (Black Swans)...")
z_scores = np.abs(stats.zscore(returns))
outliers_mask = z_scores > 3
outliers = returns[outliers_mask]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(x=returns, ax=axes[0], color='lightgray', flierprops=dict(markerfacecolor='red', marker='o', markersize=5))
axes[0].set_title("Distribution Boxplot (Red dots are Outliers)")
axes[1].plot(returns.index, returns, color='lightblue', alpha=0.5, label='Normal Days')
axes[1].scatter(outliers.index, outliers, color='red', s=20, label=f'Extreme Events (Z > 3): {len(outliers)} Days')
axes[1].set_title("Historical Black Swans")
axes[1].legend()
plt.tight_layout()
plt.show()

# ==========================================
# 4. Advanced Feature Engineering
# ==========================================
df['price_velocity'] = df['log_ret'].diff()

if 'close_usd' in df.columns:
    df['log_ret_usd'] = np.log((df['close_usd'] + 1e-6) / (df['close_usd'].shift(1) + 1e-6))
    df['price_velocity_usd'] = df['log_ret_usd'].diff()

df['Volume_SMA_50'] = df['volume'].rolling(window=50).mean()
df['RVOL_50'] = (df['volume'] / (df['Volume_SMA_50'] + 1e-9)).clip(upper=5.0)

df['day_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 7)
df['day_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 7)

# مؤشرات الـ EMA الأساسية
for period in [9, 21, 50]:
    ema_col = f'EMA_{period}'
    df[ema_col] = df['close'].ewm(span=period).mean()
    df[f'dist_EMA_{period}'] = (df['close'] - df[ema_col]) / (df[ema_col] + 1e-9)

# --- إضافة المؤشرات الخاصة بك (Trend Confirmation) ---
df['EMA_10'] = df['close'].ewm(span=10).mean()
df['EMA_10_Trend'] = (df['EMA_10'] > df['EMA_10'].shift(1)).astype(int)
df['EMA_10_Slope'] = (df['EMA_10'] - df['EMA_10'].shift(1)) / (df['EMA_10'].shift(1) + 1e-9)
# -----------------------------------------------------

delta = df['close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
df['RSI'] = 100 - (100 / (1 + gain/(loss + 1e-9)))

macd = df['close'].ewm(span=12).mean() - df['close'].ewm(span=26).mean()
df['MACD_Hist'] = macd - macd.ewm(span=9).mean()

tr = pd.concat([df['high'] - df['low'], 
                np.abs(df['high'] - df['close'].shift()), 
                np.abs(df['low'] - df['close'].shift())], axis=1).max(axis=1)
df['ATR_pct'] = tr.rolling(14).mean() / (df['close'] + 1e-9)

# ==========================================
# 5. Target Engineering (The Forward 5D Mean)
# ==========================================
print("\n🎯 Engineering Forward Target (Zero-Lag)...")
df['Future_5D_Mean'] = df['close'].rolling(window=5).mean().shift(-5)
df['Target'] = (df['Future_5D_Mean'] > df['close']).astype(int)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=['Future_5D_Mean', 'Target'], inplace=True)
df.dropna(inplace=True)

final_features = [
    'log_ret', 'price_velocity', 'RVOL_50', 
    'day_sin', 'day_cos', 
    'dist_EMA_9', 'dist_EMA_21', 'dist_EMA_50', 
    'RSI', 'MACD_Hist', 
    'Interest_Rate', 'IR_Change', 'ATR_pct',
    'EMA_10_Trend', 'EMA_10_Slope' # دمج ذكاء الـ EMA
]
X = df[final_features].values
y = df['Target'].values

# ==========================================
# 6. ML Validation & Stacking (Optuna 100)
# ==========================================
split = int(len(X) * 0.80)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

tscv = TimeSeriesSplit(n_splits=5)

def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state': 42,
        'n_jobs': -1
    }
    model = xgb.XGBClassifier(**params)
    score = cross_val_score(model, X_train_scaled, y_train, cv=tscv, scoring='accuracy').mean()
    return score

print("\n🚀 Tuning XGBoost (100 Trials)...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=100) 
best_xgb_params = study_xgb.best_params

def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    model = lgb.LGBMClassifier(**params)
    score = cross_val_score(model, X_train_scaled, y_train, cv=tscv, scoring='accuracy').mean()
    return score

print("\n🚀 Tuning LightGBM (100 Trials)...")
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=100)
best_lgb_params = study_lgb.best_params

print("\n⚙️ Training Final Stacked Model with Adjusted Bias...")
best_xgb = xgb.XGBClassifier(**best_xgb_params, random_state=42, n_jobs=-1)
best_lgb = lgb.LGBMClassifier(**best_lgb_params, random_state=42, n_jobs=-1, verbose=-1)

base_models = [('xgb', best_xgb), ('lgbm', best_lgb)]

# تعديل الأوزان لإجبار الموديل على الانتباه للهبوط (0)
final_logic = LogisticRegression(C=0.5, class_weight={0: 1.5, 1: 1.0})

kf_no_shuffle = KFold(n_splits=5, shuffle=False)
stack_model = StackingClassifier(estimators=base_models, final_estimator=final_logic, cv=kf_no_shuffle)
stack_model.fit(X_train_scaled, y_train)

# ==========================================
# 7. Evaluation & Threshold Shifting
# ==========================================
y_probs = stack_model.predict_proba(X_test_scaled)[:, 1]

# عتبة القرار (Risk Management): البيع فوراً لو التأكد من الصعود أقل من 55%
y_pred_aggressive = (y_probs >= 0.55).astype(int)
acc = accuracy_score(y_test, y_pred_aggressive) * 100

print(f"\nREAL QUANT ACCURACY (Aggressive Target & Bias Control): {acc:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_aggressive, target_names=['Down Trend (0)', 'Up Trend (1)']))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_aggressive)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn', xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'], linewidths=1, linecolor='black')
plt.title(f"EGX360 Risk-Adjusted Model\nAccuracy: {acc:.2f}%", fontsize=12, fontweight='bold')
plt.xlabel('Predicted Direction')
plt.ylabel('Actual Direction')
plt.show()

# ==========================================
# 8. Signals Visualization (User's Strategy Chart)
# ==========================================
test_dates = pd.Series(df.index[split:]).reset_index(drop=True)
test_close = df['close'].iloc[split:].reset_index(drop=True)
test_ema10 = df['EMA_10'].iloc[split:].reset_index(drop=True)

results_df = pd.DataFrame({
    'timestamp': test_dates,
    'close': test_close,
    'EMA_10': test_ema10,
    'Actual_Trend': y_test,
    'Predicted_Trend': y_pred_aggressive,
    'Probability': y_probs
})

viz_df = results_df.tail(150)
fig_test = go.Figure()
fig_test.add_trace(go.Scatter(x=viz_df['timestamp'], y=viz_df['close'], mode='lines', name='Close Price', line=dict(color='rgba(128,128,128,0.5)', width=1.5)))
fig_test.add_trace(go.Scatter(x=viz_df['timestamp'], y=viz_df['EMA_10'], mode='lines', name='EMA 10 (Trend Filter)', line=dict(color='blue', width=2)))

buy_signals = viz_df[viz_df['Predicted_Trend'] == 1]
sell_signals = viz_df[viz_df['Predicted_Trend'] == 0]

fig_test.add_trace(go.Scatter(x=buy_signals['timestamp'], y=buy_signals['EMA_10'], mode='markers', name='Model Signal: UP (1)', marker=dict(color='green', symbol='triangle-up', size=12, line=dict(width=1, color='darkgreen'))))
fig_test.add_trace(go.Scatter(x=sell_signals['timestamp'], y=sell_signals['EMA_10'], mode='markers', name='Model Signal: DOWN (0)', marker=dict(color='red', symbol='triangle-down', size=12, line=dict(width=1, color='darkred'))))

fig_test.update_layout(title='EGX360 Live Signals: Model Predictions vs Actual Market (Test Set)', yaxis_title='Price (EGP)', xaxis_title='Date', template='plotly_white', hovermode='x unified')
fig_test.show()

# ==========================================
# 9. Strategy Backtesting
# ==========================================
initial_capital = 10000.0
results_df['daily_return'] = results_df['close'].pct_change()

# Shift signal by 1 to prevent look-ahead bias
results_df['signal'] = results_df['Predicted_Trend'].shift(1)
results_df['strategy_return'] = results_df['daily_return'] * results_df['signal']

bt_df = results_df.dropna().copy()
bt_df['cumulative_market'] = (1 + bt_df['daily_return']).cumprod()
bt_df['cumulative_strategy'] = (1 + bt_df['strategy_return']).cumprod()

bt_df['portfolio_value'] = initial_capital * bt_df['cumulative_strategy']
bt_df['bh_value'] = initial_capital * bt_df['cumulative_market']

final_portfolio = bt_df['portfolio_value'].iloc[-1]
final_bh = bt_df['bh_value'].iloc[-1]
strategy_profit_pct = ((final_portfolio - initial_capital) / initial_capital) * 100
market_profit_pct = ((final_bh - initial_capital) / initial_capital) * 100

print("-" * 50)
print(f"1) Buy & Hold (Market Baseline): {final_bh:,.2f} EGP ({market_profit_pct:.2f}%)")
print(f"2) EGX360 Strategy (Aggressive): {final_portfolio:,.2f} EGP ({strategy_profit_pct:.2f}%)")
print("-" * 50)

fig_bt = go.Figure()
fig_bt.add_trace(go.Scatter(x=bt_df['timestamp'], y=bt_df['portfolio_value'], mode='lines', name='EGX360 Strategy', line=dict(color='green', width=2.5)))
fig_bt.add_trace(go.Scatter(x=bt_df['timestamp'], y=bt_df['bh_value'], mode='lines', name='Buy & Hold Baseline', line=dict(color='gray', width=1.5, dash='dash')))
fig_bt.update_layout(title='EGX360 Financial Backtest: Portfolio Growth (10,000 EGP Initial)', yaxis_title='Portfolio Value (EGP)', xaxis_title='Date', template='plotly_white', hovermode='x unified')
fig_bt.show()

joblib.dump(stack_model, "EGX360_Master_Model.pkl")
joblib.dump(scaler, "EGX360_Master_Scaler.pkl")
print("✅ Model and Scaler saved successfully!")